In [2]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from read_data import save_files

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [3]:
def read_data():
    return [
        pd.read_csv('../data/01-starting_data/development_data/awards_players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/coaches.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players.csv'),
        pd.read_csv('../data/01-starting_data/development_data/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/series_post.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams.csv'),
        pd.read_csv('../data/01-starting_data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

Section where we select relevant data and filter out invariant or irrelevant columns 

In [4]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=['firstseason', 'lastseason', 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'divID', 'arena', 'name', 'seeded', 
                        'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB',
                        'homeL', 'homeW', 'awayW', 'awayL', 'min', 'attend'])
teams_post = teams_post.drop(columns=['lgID'])

In [5]:
save_files("02-data_selection", 
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"], 
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/02-data_selection
Data coaches saved to ../data/02-data_selection
Data players saved to ../data/02-data_selection
Data players_teams saved to ../data/02-data_selection
Data series_post saved to ../data/02-data_selection
Data teams saved to ../data/02-data_selection
Data teams_post saved to ../data/02-data_selection


### Data Preparation

Section where we treat cases like non-existing values, outliers, etc.

In [6]:
players.rename(columns={'bioID': 'playerID'}, inplace=True)

In [7]:
players['pos'].fillna(0, inplace=True)
players['pos'].replace('G', 1, inplace=True)
players['pos'].replace('F-G', 2, inplace=True)
players['pos'].replace('G-F', 2, inplace=True)
players['pos'].replace('F-C', 3, inplace=True)
players['pos'].replace('C-F', 3, inplace=True)
players['pos'].replace('F', 4, inplace=True)
players['pos'].replace('C', 5, inplace=True)
players['pos'] = players['pos'].astype(int)

/tmp/ipykernel_17141/1703103431.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  players['pos'].fillna(0, inplace=True)
/tmp/ipykernel_17141/1703103431.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  players['pos'].replace('C', 5, inplace=True)


In [8]:
teams['firstRound'].fillna(0, inplace=True)
teams['firstRound'].replace('L', 0, inplace=True)
teams['firstRound'].replace('W', 1, inplace=True)
teams['firstRound'] = teams['firstRound'].astype(int)

teams['semis'].fillna(0, inplace=True)
teams['semis'].replace('L', 0, inplace=True)
teams['semis'].replace('W', 1, inplace=True)
teams['semis'] = teams['semis'].astype(int)

teams['finals'].fillna(0, inplace=True)
teams['finals'].replace('L', 0, inplace=True)
teams['finals'].replace('W', 1, inplace=True)
teams['finals'] = teams['finals'].astype(int)

/tmp/ipykernel_17141/2891936986.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  teams['firstRound'].fillna(0, inplace=True)
/tmp/ipykernel_17141/2891936986.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  teams['firstRound'].replace('W', 1, inplace=True)
/tmp/ipykernel_17141/2891936986.py:6: 

In [9]:
save_files("03-data_preparation",
           ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
           [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

Data awards_players saved to ../data/03-data_preparation
Data coaches saved to ../data/03-data_preparation
Data players saved to ../data/03-data_preparation
Data players_teams saved to ../data/03-data_preparation
Data series_post saved to ../data/03-data_preparation
Data teams saved to ../data/03-data_preparation
Data teams_post saved to ../data/03-data_preparation


### Feature Engineering

Section where we create and/or simplify existing metrics to aid the prediction model

In [10]:
teams['win_loss_ratio'] = round(teams['won'] / (teams['won'] + teams['lost']),3)

#Offensive Rating, Defensive Rating, Net Rating (overall)
teams['ORtg'] = round(teams['o_pts'] / (teams['o_fga'] + teams['o_fta']*0.44 - teams['o_oreb'] + teams['o_to']),3)
teams['DRtg'] = round(teams['d_pts'] / (teams['d_fga'] + teams['d_fta']*0.44 - teams['d_oreb'] + teams['d_to']),3)
teams['NRtg'] = round(teams['ORtg'] - teams['DRtg'],3)

#True Shooting Percentage
teams['TSht'] = round(teams['o_pts'] / (2 * (teams['o_fga'] + 0.44 * teams['o_fta'])),3)

#Rebound Percentages
teams['OReb'] = round(teams['o_oreb'] / (teams['o_oreb'] + teams['d_dreb']),3)
teams['DReb'] = round(teams['d_dreb'] / (teams['d_dreb'] + teams['o_oreb']),3)
teams['Reb'] = round(teams['OReb'] / (teams['DReb'] + teams['OReb']),3)

#3Point Rate
teams['3PtR'] = round(teams['o_3pm'] / teams['o_3pa'],3) 

#Points per Game
teams['PPG'] = round(teams['o_pts'] / teams['GP'],3)

teams.drop(columns=['o_pts', 'o_fga', 'o_fta', 'o_oreb', 'o_to', 'd_pts', 'd_fga', 'd_fta', 'd_oreb', 'd_to', 'd_dreb', 'o_3pm', 'o_3pa'], inplace=True)

#TODO - could build Pace metric using series_post df

teams['playoff_qualification'] = teams['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
teams.drop(columns=['playoff'], inplace=True)

In [11]:
#True Shooting Percentage
players_teams['TSht'] = round(players_teams['points'] / (2 * (players_teams['fgAttempted'] + 0.44 * players_teams['ftAttempted'])),3)

#Assist to Turnover Ratio
players_teams['AstTO'] = round(players_teams['assists'] / players_teams['turnovers'],3)
players_teams['AstTO'].replace([np.inf, -np.inf], 0, inplace=True)

#Rebound Percentages
players_teams['RebP'] = round(players_teams['rebounds'] / players_teams['minutes'], 3)
players_teams['ORebP'] = round(players_teams['oRebounds'] / players_teams['minutes'], 3)
players_teams['DRebP'] = round(players_teams['dRebounds'] / players_teams['minutes'], 3)

#Steal and Block Rates
players_teams['StlR'] = round(players_teams['steals'] / players_teams['minutes'],3)
players_teams['BlkR'] = round(players_teams['blocks'] / players_teams['minutes'],3)

#Points per Game
players_teams['PPG'] = round(players_teams['points'] / players_teams['GP'],3)

players_teams.drop(columns=['fgAttempted', 'ftAttempted', 'oRebounds', 'dRebounds'], inplace=True)

/tmp/ipykernel_17141/2176516666.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  players_teams['AstTO'].replace([np.inf, -np.inf], 0, inplace=True)


In [12]:
awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')
players_teams = players_teams.merge(awards_count, on=['playerID', 'year'], how='left')
players_teams['num_awards'] = players_teams['num_awards'].fillna(0)

In [13]:
column_mapping = {
    'minutes': 'MIN', 
    'points': 'PTS', 
    'oRebounds': 'ORB', 
    'dRebounds': 'DRB',
    'assists': 'AST', 
    'steals': 'STL', 
    'blocks': 'BLK', 
    'turnovers': 'TO', 
    'fouls': 'PF'
}

for old_col, new_col in column_mapping.items():
    players_teams[new_col] = players_teams.get(old_col, 0)

players_teams = per.calculate_per(players_teams)

league_average = players_teams['uPER'].mean()
players_teams['PER'] = players_teams['uPER'] * (15 / league_average)
players_teams.drop(column='uPER')


NameError: name 'per' is not defined

In [ ]:
save_files("04-feature_engineering",
            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])

### Add Year 11 info

In [14]:
def read_data11():
    return [
        pd.read_csv('../data/01-starting_data/challenge/coaches.csv'),
        pd.read_csv('../data/01-starting_data/challenge/players_teams.csv'),
        pd.read_csv('../data/01-starting_data/challenge/teams.csv')
    ]

coaches11, players_teams11, teams11 = read_data11()

coaches11 = coaches11.drop(columns=['lgID'])
players_teams11 = players_teams11.drop(columns=['lgID'])
teams11 = teams11.drop(columns=['lgID', 'franchID', 'arena', 'name'])

teams = pd.concat([teams, teams11], ignore_index=True)
players_teams = pd.concat([players_teams, players_teams11], ignore_index=True)
coaches = pd.concat([coaches, coaches11], ignore_index=True)


In [15]:
def shift_performance(data, naValueMode, unvariant, columns):
    """
    Shifts the performance metrics for next year
    Fills non-existing data with the mean of the year
    
    naValueMode- 0: fill with 0, 1: fill with 15% quantile, 2: fill with mean
    0 - values must be 0 due to context
    1 - rookie values (below average) (if there is no data, player/coach is rookie)
    2 - average values
    """
    #Shift data
    for column in columns:
        data = data.assign(**{column: data.groupby(unvariant)[column].shift(1)})
    #drop data from year 1
    data = data[data['year'] != 1]
    
    #Fill missing data with appropriate values
    for column in columns:

        if column == 'playoff_qualification':
            data[column] = data[column].fillna(0)
            continue

        year_data_mean = data.groupby('year')[column].mean()
        year_data_quantile = data.groupby('year')[column].quantile(0.15) if column not in ['rank', 'L', 'confL', 'homeL', 'lost_x', 'lost_y', 'post_losses'] else data.groupby('year')[column].quantile(0.85)
        for year in data['year'].unique():
            if naValueMode == 0:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(0)
            elif naValueMode == 1:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(round(year_data_quantile[year],2))
            elif naValueMode == 2:
                data.loc[data['year'] == year, column] = data.loc[data['year'] == year, column].fillna(round(year_data_mean[year],2))

    return data

awards_players['year'] = awards_players['year'] + 1
coaches = shift_performance(coaches, 1,['coachID'], [column for column in coaches.columns if column not in ["coachID","year","tmID", 'stint']])
players_teams = shift_performance(players_teams, 1,['playerID'], [column for column in players_teams.columns if column not in ["playerID","year","stint","tmID"]])
series_post['year'] = series_post['year'] + 1
teams_post['year'] = teams_post['year'] + 1
teams = shift_performance(teams, 1, ['tmID'], [column for column in teams.columns if column not in ['year', 'tmID', 'confID', 'confIDbin', 'playoff']])

#for debug
#save_files("05-data_shift",
#            ["awards_players", "coaches", "players", "players_teams", "series_post", "teams", "teams_post"],
#            [awards_players, coaches, players, players_teams, series_post, teams, teams_post])


Building the final data based on player data and not team overalls is aiding on reducing error, but is decreasing accuracy too

In [16]:
# dropColumns = [column for column in teams.columns if column not in ['year', 'tmID', 'confID', 'rank', 'win_loss_ratio', 'ORtg', 'DRtg', 'NRtg', 'TSht', 'OReb', 'DReb', 'Reb', '3PtR', 'PPG',
#        'playoff_qualification']]
# teams.drop(columns=dropColumns, inplace=True)

### Data Merging

Section responsible for merging all tables, in a format ready to feed the model

In [25]:
#team metrics
data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum',

    'TSht': 'mean',
    'AstTO': 'mean',
    'RebP': 'mean',
    'ORebP': 'mean',
    'DRebP': 'mean',
    'StlR': 'mean',
    'BlkR': 'mean',
    'PPG': 'mean',
    'PER': 'mean',

    'num_awards': 'sum'
}).reset_index()
for column in ['TSht', 'AstTO', 'RebP', 'ORebP', 'DRebP', 'StlR', 'BlkR', 'PPG']:
    player_stats[column] = round(player_stats[column],3)

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()
data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")

data.columns

KeyError: "Column(s) ['PER'] do not exist"

In [ ]:
#write df to csv
save_files("05-processed_data", ["processed_data"], [data])

data.fillna(0, inplace=True)

print(f"final data columns {data.columns}")

In [ ]:
numeric_data = data.select_dtypes(include=[np.number])

# Check for infinite values
print("Infinite values:")
for col in numeric_data.columns:
    inf_values = numeric_data[np.isinf(numeric_data[col])]
    if not inf_values.empty:
        print(f"Column: {col}")
        print(inf_values)

# Check for extremely large values and print the exact values and columns
threshold = 1e308  # This is close to the maximum value for float64
print("\nExtremely large values:")
for col in numeric_data.columns:
    large_values = numeric_data[numeric_data[col] > threshold]
    if not large_values.empty:
        print(f"Column: {col}")
        print(large_values)

# Handle problematic values
# Replace infinite and extremely large values with NaN in numeric columns
numeric_data.replace([np.inf, -np.inf], np.nan, inplace=True)
numeric_data[numeric_data > threshold] = np.nan

# Round all numeric values to 3 decimal places
numeric_data = numeric_data.round(3)

# Update the original data with the cleaned numeric data
data.update(numeric_data)

### Model training

In [ ]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR

#### Initialization 

In [ ]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['playoff_qualification', 'tmID', 'confID']]
target_column = 'playoff_qualification'

#Create model
#model = DecisionTreeClassifier(random_state=21) #acc: 0.55 error: 0.59 || acc: 0.59  error: 0.55
model = RandomForestClassifier(random_state=21) #acc: 0.60 error: 0.43 || acc: 0.64  error: 0.44
#model = LogisticRegression(random_state=21)     #acc: 0.64 error: 0.39 || acc: 0.57  error: 0.44
#model = SVR()                                   #acc: 0.62 error: 0.44 || acc: 0.60  error: 0.45

data.columns

#### Year training cycle

In [ ]:
for year in sorted(data['year'].unique()):  # Start from the second year (with )
    # Separate train and test data
    year_span = 9
    train_data = data[data['year'] <= year] if year < (year_span - 1) else data[(data['year'] <= year) & (data['year'] >= year - (year_span - 1))]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip (redundant?)
    if test_data.empty:
        continue

    y_test = test_data[target_column]

    results_df = pd.DataFrame()
    for confID in data['confID'].unique():
        conf_train_data = train_data[train_data['confID'] == confID]
        conf_test_data = test_data[test_data['confID'] == confID]
    
        
        confIDs = conf_test_data['confID'].values
        teamIDs = conf_test_data['tmID'].values
    
        conf_X_train = conf_train_data[feature_columns]
        conf_y_train = conf_train_data[target_column]
    
        conf_X_test = conf_test_data[feature_columns]
        conf_y_test = conf_test_data[target_column]
        
        # Standardize the data
        scaler = StandardScaler()
        conf_X_train = scaler.fit_transform(conf_X_train)
        conf_X_test = scaler.transform(conf_X_test)
    
        # Train the model
        model.fit(conf_X_train, conf_y_train)
        if (year == 10):
            feature_importances =  pd.DataFrame({
                'feature': feature_columns,
                'importance': model.feature_importances_
            })
            feature_importances.sort_values(by="importance", ascending=False).to_csv(f'../data/06-results/importances_{confID}.csv', index=False)
    
        # Make predictions on the test set
        y_pred_proba = model.predict_proba(conf_X_test)[:,1]
    
    
        conf_results_df = pd.DataFrame({
            'tmID': teamIDs,
            'confID': confIDs,
            'raw': np.round(y_pred_proba, 3)
        })
    
        results_df = pd.concat([results_df, conf_results_df])
        results_df.sort_values(by='tmID', ascending=True, inplace=True)

    results_df['Playoff'] = (results_df['raw']* 8 / results_df['raw'].sum()).round(2)
    results_df['Label'] = np.zeros_like(results_df['Playoff'])

    #get the top 4 teams of each conference
    top_teams = (
        results_df.groupby('confID', group_keys=False)
        .apply(lambda group: group.nlargest(4, 'Playoff'))
        .reset_index(drop=True)
    )
    results_df.loc[results_df['tmID'].isin(top_teams['tmID']), 'Label'] = 1

    
    y_pred = results_df['Label'] 
    y_pred_proba_norm = results_df['Playoff']

    if (year == 10):
        #results_df = results_df.drop(columns=['Label', 'confID'])
        results_df.to_csv('../data/06-results/results.csv', index=False)
    # Calculate accuracy and error

    if year < 10:
        accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

        error_array = np.abs(y_pred_proba_norm - y_test.values)
        error_score = round(sum(error_array),2)
        error_scores.append(error_score)

        # Output results for each year
        print(f"Year {year} -> {year + 1}:")
        print(f"Results: \n predict: \n {results_df['Playoff'].values}\n label: \t {results_df['Label'].values}\n expected: {y_test.values}\n error: \t {error_array.values}")
        print(f"  Accuracy: {accuracy_scores[-1]}")
        print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)}")
        print("\n")

/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))
/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))


Year 2 -> 3:
Results: 
 predict: 
 [0.77 0.64 0.25 0.58 0.24 0.86 0.68 0.32 0.86 0.39 0.37 0.34 0.64 0.36
 0.39 0.34]
 label: 	 [1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0.]
 expected: [1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0.]
 error: 	 [0.23 0.36 0.25 0.42 0.24 0.14 0.32 0.32 0.14 0.39 0.37 0.34 0.36 0.36
 0.61 0.34]
  Accuracy: 1.0
  Error: 	 5.19 / 16


Year 3 -> 4:
Results: 
 predict: 
 [0.94 0.07 0.3  0.15 1.06 1.03 0.97 0.13 0.98 0.21 0.1  0.32 0.7  1.05]
 label: 	 [1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 0. 1. 1. 1.]
 expected: [1. 0. 0. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 1.]
 error: 	 [0.06 0.07 0.3  0.15 0.06 0.03 0.03 0.13 0.02 0.21 0.1  0.32 0.3  0.05]
  Accuracy: 0.93
  Error: 	 1.83 / 14




/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))
/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))


Year 4 -> 5:
Results: 
 predict: 
 [0.79 0.87 1.03 0.89 0.58 0.85 0.67 0.57 0.11 0.8  0.18 0.6  0.05]
 label: 	 [1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 0.]
 expected: [1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0.]
 error: 	 [0.21 0.13 0.03 0.11 0.58 0.15 0.33 0.57 0.11 0.2  0.18 0.6  0.05]
  Accuracy: 0.92
  Error: 	 3.25 / 13


Year 5 -> 6:
Results: 
 predict: 
 [0.27 1.03 0.82 0.14 0.19 0.99 0.56 0.88 0.51 0.89 0.07 0.99 0.67]
 label: 	 [0. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1.]
 expected: [0. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1.]
 error: 	 [0.27 0.03 0.18 0.14 0.19 0.01 0.44 0.12 0.51 0.11 0.07 0.01 0.33]
  Accuracy: 1.0
  Error: 	 2.41 / 13




/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))
/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))


Year 6 -> 7:
Results: 
 predict: 
 [0.03 0.25 1.   0.52 0.85 1.03 0.65 0.09 0.9  0.41 1.   0.06 0.89 0.31]
 label: 	 [0. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 1. 0.]
 expected: [0. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 1. 0.]
 error: 	 [0.03 0.25 0.   0.48 0.15 0.03 0.35 0.09 0.1  0.41 0.   0.06 0.11 0.31]
  Accuracy: 1.0
  Error: 	 2.37 / 14


Year 7 -> 8:
Results: 
 predict: 
 [0.06 0.95 0.96 0.79 0.89 0.96 0.16 0.07 0.44 0.96 0.16 0.78 0.83]
 label: 	 [0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1.]
 expected: [0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1.]
 error: 	 [0.06 0.05 0.04 0.21 0.11 0.04 0.16 0.07 0.44 0.04 0.16 0.22 0.17]
  Accuracy: 1.0
  Error: 	 1.77 / 13




/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))
/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))


Year 8 -> 9:
Results: 
 predict: 
 [0.16 0.1  0.96 0.99 0.15 0.97 0.11 0.11 0.72 0.83 0.99 0.94 0.72 0.23]
 label: 	 [0. 0. 1. 1. 0. 1. 0. 0. 1. 1. 1. 1. 1. 0.]
 expected: [0. 0. 1. 1. 0. 1. 0. 0. 1. 1. 1. 1. 1. 0.]
 error: 	 [0.16 0.1  0.04 0.01 0.15 0.03 0.11 0.11 0.28 0.17 0.01 0.06 0.28 0.23]
  Accuracy: 1.0
  Error: 	 1.74 / 14


Year 9 -> 10:
Results: 
 predict: 
 [0.11 0.33 0.94 0.92 0.93 0.87 0.14 0.93 0.14 0.71 0.89 0.94 0.12]
 label: 	 [0. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0.]
 expected: [0. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0.]
 error: 	 [0.11 0.33 0.06 0.08 0.07 0.13 0.14 0.07 0.14 0.29 0.11 0.06 0.12]
  Accuracy: 1.0
  Error: 	 1.71 / 13




/tmp/ipykernel_17141/691248934.py:61: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.nlargest(4, 'Playoff'))


### End Results

In [ ]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores) / len(accuracy_scores):.2f}")
print(f"  Average Error: \t {round(sum(error_scores) / len(error_scores),2)}")

Accuracy  [1.0, 0.93, 0.92, 1.0, 1.0, 1.0, 1.0, 1.0]
Error 	 [5.19, 1.83, 3.25, 2.41, 2.37, 1.77, 1.74, 1.71]

Average Performance Over All Years:
  Average Accuracy: 0.98
  Average Error: 	 2.53


### Calculate Player Efficiency Rating (PER)

In [24]:
import os
import pandas as pd
import read_data as rd
import player_efficiency_rating as per

# Step 1: Load the data
print("Loading data...")
tables = rd.read_data()

# Step 2: Check for and load player statistics data
print("Calculating Player Efficiency Rating (PER)...")
if "players_teams" not in tables:
    raise KeyError("Table 'players_teams' not found in loaded data!")
players_stats = tables["players_teams"]

# Step 3: Map the required columns for PER calculation
column_mapping = {
    'minutes': 'MIN', 
    'points': 'PTS', 
    'oRebounds': 'ORB', 
    'dRebounds': 'DRB',
    'assists': 'AST', 
    'steals': 'STL', 
    'blocks': 'BLK', 
    'turnovers': 'TO', 
    'fouls': 'PF'
}

for old_col, new_col in column_mapping.items():
    players_stats[new_col] = players_stats.get(old_col, 0)

# Display sample statistics
print(players_stats[['playerID', 'MIN', 'PTS', 'ORB', 'DRB']].head())

# Step 4: Calculate the PER
players_stats = per.calculate_per(players_stats)

# Step 5: Normalize PER to set the league average to 15
league_average = players_stats['uPER'].mean()
players_stats['PER'] = players_stats['uPER'] * (15 / league_average)

# Step 6: Define the output directory (outside 'src') and ensure it exists
output_dir = os.path.join("..", "data", "07-player_efficiency_rating")  # Move up one level
os.makedirs(output_dir, exist_ok=True)

# Step 7: Save the results to a CSV file
output_file = os.path.join(output_dir, "players_with_per.csv")
players_stats.to_csv(output_file, index=False)
print(f"Player Efficiency Rating (PER) calculated and saved successfully to '{output_file}'!")

# Step 8: Create a summary table with playerID, uPER, and PER
per_summary = players_stats[['playerID', 'uPER', 'PER']]

# Step 9: Display and save the summary table
print(per_summary.head())
summary_file = os.path.join(output_dir, "players_per_summary.csv")
per_summary.to_csv(summary_file, index=False)
print(f"Summary table saved as '{summary_file}'.")


Loading data...
Calculating Player Efficiency Rating (PER)...
     playerID  MIN  PTS  ORB  DRB
0  abrossv01w  846  343   43  131
1  abrossv01w  805  314   45  101
2  abrossv01w  792  318   44   97
3  abrossv01w  462  146   17   57
4  abrossv01w  777  304   29   78
Player Efficiency Rating (PER) calculated and saved successfully to '../data/07-player_efficiency_rating/players_with_per.csv'!
     playerID      uPER        PER
0  abrossv01w  0.509929  18.152665
1  abrossv01w  0.491677  17.502921
2  abrossv01w  0.536490  19.098188
3  abrossv01w  0.452381  16.104043
4  abrossv01w  0.491248  17.487662
Summary table saved as '../data/07-player_efficiency_rating/players_per_summary.csv'.
